<a href="https://colab.research.google.com/github/AngeloSorte/ai-agent-mcp-server-python/blob/angelosorte.github.io/MCP_DevOps_Tool_Server_Python_AI_Agent_Backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# MCP-STYLE DEVOPS TOOL SERVER (SINGLE FILE - COLAB READY)

from fastapi import FastAPI
from pydantic import BaseModel
import sqlite3
import uvicorn
import json

app = FastAPI(title="MCP DevOps Tool Server")

# -----------------------------
# TOOL REGISTRY
# -----------------------------
TOOLS = {}

def tool(name):
    def wrapper(func):
        TOOLS[name] = func
        return func
    return wrapper

# -----------------------------
# GITHUB MOCK TOOL
# -----------------------------
@tool("github_repo_info")
def github_repo_info(payload):
    repo = payload.get("repo", "unknown")
    return {
        "repo": repo,
        "stars": 128,
        "open_issues": 3,
        "status": "active"
    }

# -----------------------------
# LOG ANALYZER TOOL
# -----------------------------
@tool("log_analyzer")
def log_analyzer(payload):
    logs = payload.get("logs", "")
    errors = logs.lower().count("error")
    warnings = logs.lower().count("warn")
    return {
        "errors": errors,
        "warnings": warnings,
        "summary": "log analyzed"
    }

# -----------------------------
# SQLITE TOOL
# -----------------------------
conn = sqlite3.connect(":memory:", check_same_thread=False)
cursor = conn.cursor()
cursor.execute("CREATE TABLE users (id INTEGER, name TEXT)")
cursor.execute("INSERT INTO users VALUES (1, 'angelo'), (2, 'dev')")
conn.commit()

@tool("db_query")
def db_query(payload):
    query = payload.get("query", "SELECT * FROM users")
    try:
        res = cursor.execute(query).fetchall()
        return {"result": res}
    except Exception as e:
        return {"error": str(e)}

# -----------------------------
# MCP RUNNER
# -----------------------------
class ToolRequest(BaseModel):
    tool: str
    payload: dict

@app.post("/run")
def run_tool(req: ToolRequest):
    if req.tool not in TOOLS:
        return {"error": "tool not found"}
    return TOOLS[req.tool](req.payload)

@app.get("/tools")
def list_tools():
    return {"tools": list(TOOLS.keys())}

# -----------------------------
# START SERVER (FOR COLAB / LOCAL)
# -----------------------------
if __name__ == "__main__":
    import nest_asyncio
    import threading

    nest_asyncio.apply()

    def run():
        uvicorn.run(app, host="0.0.0.0", port=8000)

    thread = threading.Thread(target=run)
    thread.start()

    print("MCP DevOps Tool Server running on http://localhost:8000")

MCP DevOps Tool Server running on http://localhost:8000
